# 01 - Data Acquisition & Understanding

Project: **RoadEye - Traffic Sign Recognition Using HOG Features and SVM** (Machine Learning Innovative Assignment).

> Run cells top to bottom. All numbers are computed when the cell runs; nothing is typed in by hand.

In [ ]:
import sys, pathlib, json, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
from src import config, pipeline as P
np.random.seed(config.SEED)                      # reproducibility
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("RUN MODE:", config.RUN_MODE, "| seed:", config.SEED)

In [ ]:
def show(name, width=900):
    """Display a saved figure from results/figures/."""
    display(Image(filename=str(config.FIGURES_DIR / name), width=width))

def metrics(name):
    return json.loads((config.METRICS_DIR / name).read_text())

## 1. Locate and verify the dataset
The code never assumes GTSRB exists: `locate_dataset()` raises a clear error pointing to `DATASET_DOWNLOAD.md` if it is missing.

**Why GTSRB?** It is the standard, citable benchmark for multi-class traffic-sign recognition (43 classes), with official labels, per-image ROI boxes and a separate labelled test set.

In [ ]:
from src.data_loader import locate_dataset, build_metadata
info = locate_dataset()
print('class folders:', len(info['class_dirs']), '| labelled test set:', info['test_gt'] is not None)

## 2. Audit: counts, classes, sizes, corrupt files, labels, duplicates
`stage_audit()` reads **every** file (decoder check + MD5) and writes `results/metrics/dataset_audit.json`, `class_distribution.png`, `image_size_distribution.png`.

In [ ]:
audit = P.stage_audit()
summary = {k: audit[k] for k in ['run_mode','n_files_total','n_train_pool_files','n_official_test_files','n_unreadable_or_missing','file_formats','n_classes_train','class_count_min','class_count_max','imbalance_ratio_max_over_min','n_tracks_train','n_features_raw_pixels']}
display(pd.Series(summary))

In [ ]:
display(pd.Series(audit['label_problems'], name='label checks'))
display(pd.Series(audit['duplicate_files_in_train'], name='duplicate analysis'))
print('identical files shared by train pool and official test:', audit['train_test_identical_files'])
display(pd.DataFrame({'width': audit['image_width'], 'height': audit['image_height']}))
print(audit['sanity_checks_vs_official_documentation'])

## 3. Class distribution and representative images

In [ ]:
show('class_distribution.png'); show('image_size_distribution.png', 700)

**What to look for:** (i) strong class imbalance -> see Stage 5 in notebook 02/03; (ii) very different original image sizes -> a fixed-size resize is mandatory; (iii) any unreadable files or label mismatches (should be zero).

## 4. Dataset facts to document
* Source: official GTSRB archives (see `DATASET_DOWNLOAD.md`).
* Citation: Stallkamp et al., *Neural Networks* 32 (2012).
* Licence/usage: **verify on the archive page you downloaded from and record it in `LICENSE_NOTICE.md`.**
* Target: `ClassId` in {0..42}, one traffic-sign type per image.